In [ ]:
# ==============================================================
# FCRM Risk Assessment FY2025
# Metric: Centralized (1.1, 1.2-1.5, 1.7/1.8, 1.9, SD2, SD6, 3.17-3.19)
# AU: 301368 — Contact Center
# Jira Tickets:
#   1.1: FY25DATA-3452
#   1.7: FY25DATA-3461
#   1.8: FY25DATA-3462
#   3.17: FY25DATA-3479
#   3.18: FY25DATA-3480
#   3.19: FY25DATA-3481
#   SD2: FY25DATA-3445
#   SD6: FY25DATA-3449
# Work Package: [WP-XX]
# Owner: Hua
# Reviewer: [name]
# Last Updated: 2025-05-13
# Status: BUILD
# ==============================================================
#
# Description:
# Centralized notebook computing all BDE/SD metrics for the
# Contact Center assessment unit (301368). Covers risk-tier
# customer counts (1.1-1.5), sanctions (1.7/1.8, 1.9), PEP (SD2),
# new customers (SD6), UTR (3.17), SARs (3.18), and LCTRs (3.19).
#
# Data Sources:
#   - ra_fy_2025.cif_accounts_fy25 (Rahona)
#   - ra_fy_2025.cif_personal_fy25 / cif_non_personal_fy25 (Rahona)
#   - rafy2025_centralized.CRR_Scorable_Cust_CA (Rahona)
#   - rafy2025_centralized.customer_scorable_rated_ca (Rahona)
#   - rafy2025_centralized.customer_scorable_unrated_ca (Rahona)
#   - ra_adido_2025.true_sanction_match_2025 (ADIDO)
#   - ra_adido_2025.customer_sanctioned_blocked_property_2025 (ADIDO)
#   - ra_adido_2025.pep_list_2025_exploded (ADIDO)
#   - rafy2025_centralized.cde_sd_6_1yr_fy2025 (Rahona)
#   - rafy2025_centralized.td_utr_cde_3_17_2025_cust_details (Rahona)
#   - rafy2025_centralized.td_sar_cde_3_18_2025 (Rahona)
#   - rafy2025_centralized.cde_3_19_lctr (Rahona)
#   - caedw.cust (CZ — JDBC)
#
# Output: Display-only (metric counts). No target table written.
# ==============================================================

In [ ]:
# ==============================================================
# Cell 2: Imports & Configuration
# ==============================================================
from pyspark.sql.functions import *

# JDBC connection strings
czJdbcUrl = "jdbc:sqlserver://p3001-eastus2-asql-2.database.windows.net:1433;database=eda-akora2-aaecz-corporatepoolprd;loginTimeout=10;"
srzJdbcURL = "jdbc:sqlserver://p3001-eastus2-asql-3.database.windows.net:1433;database=eda-akora2-aaed1-srzpoolprd;loginTimeout=10"
azJdbcURL = "jdbc:sqlserver://p3006-eastus2-asql-1.database.windows.net:1433;database=eda-akora-aaaz-CAGAML00BI0001ClusterPRD;loginTimeout=10"
izJdbcUrl = "jdbc:sqlserver://p3004-eastus2-asql-32.database.windows.net:1433;database=eda-akora-aaicz-icz00001poolprd;loginTimeout=10"

jdbcUsername = dbutils.secrets.get(scope="aaaz-base", key="SP_ADB_AAAZ_CAGAML00BI0001_PRD_AppID")
jdbcPassword = dbutils.secrets.get(scope="aaaz-base", key="SP_ADB_AAAZ_CAGAML00BI0001_PRD_PWD")

connectionProperties = {
    "AADSecurePrincipalID" : jdbcUsername,
    "AADSecurePrincipalSecret" : jdbcPassword,
    "driver" : "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "authentication" : "ActiveDirectoryServicePrincipal",
    "fetchsize" : "10"
}

In [ ]:
# ==============================================================
# Cell 3: [LINEAGE] Source Definition
# ==============================================================
# AU Universe:
#   Table: rafy2025_centralized.contact_centre_au_20251031
#   Built from: ra_fy_2025.cif_accounts_fy25
#     INNER JOIN cif_personal_fy25 (type=0) / cif_non_personal_fy25 (type=1)
#   Filters: bank_num=4, aplictn_id IN (ACS, VSA),
#            ifw_effective_date <= 20251031, customr_status = '00'
#   Logic: SELECT DISTINCT acc.customr_num, acc.customr_type (no GROUP BY dedup)
#   Pull date: 2025-11-01
#
# CAEDW Customer Dimension:
#   Source: caedw.cust (via JDBC — CZ pool)
#   Fields: cust_id, cust_no, cust_type_mn
#   Dedup: row_number() OVER (PARTITION BY cust_id ORDER BY to_dt DESC) = 1
#
# Scored/Rated/Unrated Tables (Rahona):
#   - rafy2025_centralized.CRR_Scorable_Cust_CA
#   - rafy2025_centralized.customer_scorable_rated_ca
#   - rafy2025_centralized.customer_scorable_unrated_ca
#   CIF key derivation: cust_no = last 9 chars of v_entity_id,
#     cust_type_mn = 'N' if position 8 == '1' else 'P'
#
# ADIDO Sources:
#   - ra_adido_2025.true_sanction_match_2025 (key: Customer#)
#   - ra_adido_2025.customer_sanctioned_blocked_property_2025
#   - ra_adido_2025.pep_list_2025_exploded (key: ENTITY)
#
# Centralized Metric Sources:
#   - rafy2025_centralized.cde_sd_6_1yr_fy2025 (key: v_entity_id)
#   - rafy2025_centralized.td_utr_cde_3_17_2025_cust_details (key: cust_no)
#   - rafy2025_centralized.td_sar_cde_3_18_2025 (key: STR_Admin_Matched_CustomerID)
#   - rafy2025_centralized.cde_3_19_lctr / cde_3_19_lctr_tds_cowen (key: beneficiaries_clientNumber)

In [ ]:
%sql
-- ==============================================================
-- Cell 4a: Data Extraction — Build AU Universe
-- SELECT DISTINCT acc.customr_num, acc.customr_type (original logic)
-- ==============================================================
CREATE OR REPLACE TABLE rafy2025_centralized.contact_centre_au_20251031
USING DELTA
AS
SELECT DISTINCT acc.customr_num, acc.customr_type
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_personal_fy25 pers
  ON acc.customr_num = pers.customr_num
  AND acc.customr_bank_num = pers.customr_bank_num
  AND acc.customr_type = pers.customr_type
WHERE acc.customr_bank_num = 4
  AND acc.customr_type = 0
  AND acc.aplictn_id IN ('ACS', 'VSA')
  AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
  AND pers.customr_status = '00'
UNION
SELECT DISTINCT acc.customr_num, acc.customr_type
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_non_personal_fy25 npers
  ON acc.customr_num = npers.customr_num
  AND acc.customr_bank_num = npers.customr_bank_num
  AND acc.customr_type = npers.customr_type
WHERE acc.customr_bank_num = 4
  AND acc.customr_type = 1
  AND acc.aplictn_id IN ('ACS', 'VSA')
  AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
  AND npers.customr_status = '00'

In [ ]:
# ==============================================================
# Cell 4b: Data Extraction — Load AU, derive CIF keys, enrich with CAEDW
# ==============================================================

# Load AU universe
au = spark.table("rafy2025_centralized.contact_centre_au_20251031")

# Derive standardized CIF keys:
#   cust_no     = 9-digit zero-padded customer number
#   cust_type_mn = 'N' (non-personal, type=1), 'P' (personal, type=0)
au_cif = (
    au
    .withColumn('cust_no', lpad(col('customr_num'), 9, '0'))
    .withColumn('cust_type_mn', when(col('customr_type') == '1', 'N').otherwise('P'))
)

# CAEDW customer dimension via JDBC (latest record per cust_id)
cust_query = """(
    SELECT cust_id, cust_no, cust_type_mn
    FROM (
        SELECT *, row_number() OVER (PARTITION BY cust_id ORDER BY to_dt DESC) AS row_num
        FROM caedw.cust
    ) c
    WHERE c.row_num = 1
) t"""
df_cust = spark.read.jdbc(url=czJdbcUrl, table=cust_query, properties=connectionProperties).cache()

# Enrich AU with CAEDW (left join retains all AU records)
au_enriched = (
    au_cif
    .join(df_cust,
          (au_cif.cust_no == df_cust.cust_no) & (au_cif.cust_type_mn == df_cust.cust_type_mn),
          how='left')
    .drop(df_cust.cust_no).drop(df_cust.cust_type_mn)
)

In [ ]:
# ==============================================================
# Cell 4c: Data Extraction — Load reference tables (once, reused below)
# ==============================================================

# Scored customers (for metric 1.1)
scored_src = spark.table("rafy2025_centralized.CRR_Scorable_Cust_CA")
scored_cif = (
    scored_src.filter(col('v_entity_id').startswith('CIF'))
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)

# Rated customers (for 1.2, 1.3, 1.4, Low part of 1.5)
rated_src = spark.table('rafy2025_centralized.customer_scorable_rated_ca')
rated_cif = (
    rated_src.filter(col('v_entity_id').startswith('CIF'))
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)
au_rated = rated_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')

# Unrated customers (for unscored part of 1.5)
unrated_src = spark.table('rafy2025_centralized.customer_scorable_unrated_ca')
unrated_cif = (
    unrated_src.filter(col('v_entity_id').startswith('CIF'))
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)
au_unrated = unrated_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')

In [ ]:
# ==============================================================
# Cell 5: [LINEAGE] Business Logic & Transformations
# ==============================================================
# All metrics below join source data with au_enriched on
# [cust_no, cust_type_mn] to scope results to this AU.
#
# BDE 1.1 — Unscored: left_anti join AU vs scored_cif
# BDE 1.2 — Tier 1/2: au_rated WHERE risk_rating IN ('Tier 1','Tier 2')
# BDE 1.3 — High Risk: au_rated WHERE risk_rating = 'High'
# BDE 1.4 — Medium:    au_rated WHERE risk_rating = 'Medium'
# BDE 1.5 — Low+Unscored: UNION of (au_rated WHERE 'Low') and au_unrated
# BDE 1.7 — True Sanctions: true_sanction_match_2025, CIF filter, countDistinct(Case#)
# BDE 1.8 — Blocked Accounts: customer_sanctioned_blocked_property_2025 (SQL)
# SD 2.0 — PEP: pep_list_2025_exploded, CIF filter, countDistinct(cust_intrl_id)
# SD 6.0 — New Customers: cde_sd_6_1yr_fy2025, CIF filter, countDistinct(cust_intrl_id)
# BDE 3.17 — UTR: td_utr_cde_3_17_2025_cust_details, countDistinct(utrid)
# BDE 3.18 — SARs: td_sar_cde_3_18_2025, countDistinct(STR_Internal_Unique_ID)
# BDE 3.19 — LCTRs: N/A for this channel
#
# CIF key derivation (applied to all v_entity_id / Customer# / ENTITY fields):
#   cust_no      = substring(field, -9, 9)         — last 9 chars
#   cust_type_mn = 'N' if char at pos 8 (or 4) is '1' else 'P'
#
# Count metric: countDistinct('cust_intrl_id') unless noted otherwise.

In [ ]:
# --- BDE 1.1: Unrated / Unscored Customers ---
print("Scored CIF distinct customers =", scored_cif.count())
m_1_1 = (
    au_enriched
    .join(scored_cif, on=["cust_no", "cust_type_mn"], how="left_anti")
    .dropDuplicates()
)
print("Metric 1.1 (CIF unscored in AU) =", m_1_1.count())

In [ ]:
# --- BDE 1.2: Tier 1/2 High Risk Customers ---
m_1_2 = au_rated.filter(col('risk_rating').isin("Tier 1", "Tier 2"))
m_1_2.agg(countDistinct('cust_intrl_id').alias('1.2')).display()

In [ ]:
# --- BDE 1.3: High Risk Customers (excluding Tier 1/2) ---
m_1_3 = au_rated.filter(col('risk_rating') == "High")
m_1_3.agg(countDistinct('cust_intrl_id').alias('1.3')).display()

In [ ]:
# --- BDE 1.4: Medium Risk Customers ---
m_1_4 = au_rated.filter(col('risk_rating') == "Medium")
m_1_4.agg(countDistinct('cust_intrl_id').alias('1.4')).display()

In [ ]:
# --- BDE 1.5: Low Risk + Unscored Customers ---
m_1_5_low = au_rated.filter(col('risk_rating') == 'Low')
m_1_5 = m_1_5_low.unionByName(au_unrated, allowMissingColumns=True)
m_1_5.agg(countDistinct('cust_intrl_id').alias('1.5')).display()

In [ ]:
# --- BDE 1.7: True Sanctions Matches ---
# Note: JIRA ticket is BDE_1.7; metric alias shows 1.8 per current numbering.
tsm_src = spark.table('ra_adido_2025.true_sanction_match_2025')
tsm_cif = (
    tsm_src.filter(col("Customer#").startswith('CIF'))
    .withColumn('cust_no', substring(col('Customer#'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('Customer#'), 4, 1) == '1', 'N').otherwise('P'))
)
display(tsm_cif)
au_tsm = tsm_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_tsm.agg(countDistinct("Case#").alias('1.8')).display()

In [ ]:
%sql
-- --- BDE 1.8: Blocked Accounts / Asset Freezes ---
SELECT * FROM ra_adido_2025.customer_sanctioned_blocked_property_2025;

In [ ]:
# --- SD 2.0: PEP (Politically Exposed Persons) ---
pep_src = spark.table("ra_adido_2025.pep_list_2025_exploded")
pep_cif = (
    pep_src.filter(col('ENTITY').startswith('CIF'))
    .withColumn('cust_no', substring(col('ENTITY'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('ENTITY'), 8, 1) == '1', 'N').otherwise('P'))
)
au_pep = pep_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_pep.agg(countDistinct('cust_intrl_id').alias('SD2')).display()
au_pep.filter(col("PEP_TYPE").isNotNull()).agg(countDistinct('cust_intrl_id').alias('SD2')).display()
sd2_summary = au_pep.groupBy("PEP_TYPE").count().orderBy("count", ascending=False)
sd2_notNull = au_pep.filter(col("PEP_TYPE").isNotNull()).groupBy("PEP_TYPE").count().orderBy("count", ascending=False)
display(sd2_summary)
display(sd2_notNull)

In [ ]:
# --- SD 6.0: Customers with TD less than 1 year ---
sd6_src = spark.table('rafy2025_centralized.cde_sd_6_1yr_fy2025')
sd6_cif = (
    sd6_src.filter(col('v_entity_id').startswith('CIF'))
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)
display(sd6_cif)
au_sd6 = sd6_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_sd6.agg(countDistinct('cust_intrl_id').alias('sd6')).display()

In [ ]:
# --- BDE 3.17: Unusual Transaction Referrals (UTR) ---
utr_src = spark.table("rafy2025_centralized.td_utr_cde_3_17_2025_cust_details")
au_utr = utr_src.join(au_enriched, on=["cust_no", "cust_type_mn"], how="inner")
display(au_utr.limit(30))
au_utr.agg(countDistinct("utrid").alias("metric_3_17_distinct")).display()
au_utr.agg(count("cust_no").alias("metric_3_17")).display()

In [ ]:
# --- BDE 3.18: SARs/STRs ---
sar_src = spark.table("rafy2025_centralized.td_sar_cde_3_18_2025")
au_sar = (
    sar_src.alias("s")
    .join(
        au_enriched.alias("a"),
        (col("s.STR_Admin_Matched_CustomerID") == col("a.cust_no")) &
        (col("s.STR_Admin_Matched_Customer_Type") == col("a.cust_type_mn")),
        "inner"
    )
)
au_sar.agg(countDistinct("STR_Internal_Unique_ID").alias("metric 3.18")).display()

In [ ]:
# --- BDE 3.19: LCTRs ---
# Not applicable for Contact Center.
print("3.19 LCTRs - Not applicable for Contact Center")

In [ ]:
# ==============================================================
# Cell 7: [LINEAGE] Output Definition
# ==============================================================
# This notebook produces display-only metric counts.
# No output tables are written. Results are recorded in the
# Excel mastersheet (SharePoint: 06 - Change Management / Evidence).
#
# Downstream consumers: Risk Assessment mastersheet (BA reconciliation).

In [ ]:
# ==============================================================
# Cell 9: Reconciliation Summary
# ==============================================================
# Print all metric counts for BA reconciliation.
print("=== Reconciliation Summary ===")
print(f"  1.1  (Unscored):        {m_1_1.count():>12,}")
print(f"  1.2  (Tier 1/2):        {m_1_2.agg(countDistinct('cust_intrl_id')).collect()[0][0]:>12,}")
print(f"  1.3  (High):            {m_1_3.agg(countDistinct('cust_intrl_id')).collect()[0][0]:>12,}")
print(f"  1.4  (Medium):          {m_1_4.agg(countDistinct('cust_intrl_id')).collect()[0][0]:>12,}")
print(f"  1.5  (Low+Unscored):    {m_1_5.agg(countDistinct('cust_intrl_id')).collect()[0][0]:>12,}")